In [ ]:
import tensorflow as tf
print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TF version: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# ── Step 1: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Step 2: Set up folder structure ───────────────────────────────────────
import os

os.makedirs('/content/models/saved_models', exist_ok=True)
os.makedirs('/content/results/metrics', exist_ok=True)
os.makedirs('/content/results/plots', exist_ok=True)

# ── Step 3: Verify dataset ─────────────────────────────────────────────────
DATASET_DIR = '/content/drive/MyDrive/breast_cancer/400X'

benign_count    = len(os.listdir(os.path.join(DATASET_DIR, 'benign')))
malignant_count = len(os.listdir(os.path.join(DATASET_DIR, 'malignant')))

print(f"Benign images    : {benign_count}")
print(f"Malignant images : {malignant_count}")
print(f"Total            : {benign_count + malignant_count}")

# ── Step 4: Data generators ────────────────────────────────────────────────
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    validation_split=0.30
)

val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.30
)

train_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True,
    seed=SEED
)

val_generator = val_test_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False,
    seed=SEED
)

test_generator = val_test_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False,
    seed=SEED + 1
)

print(f"\nTrain      : {train_generator.samples} images")
print(f"Validation : {val_generator.samples} images")
print(f"Test       : {test_generator.samples} images")
print(f"Classes    : {train_generator.class_indices}")

# ── Step 5: Build model ────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import (
    VGG16, ResNet50, DenseNet121, InceptionV3
)
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping,
    ReduceLROnPlateau, CSVLogger
)

IMG_SHAPE     = (224, 224, 3)
LEARNING_RATE = 1e-4
EPOCHS        = 30

BASE_MODELS = {
    'VGG16'       : VGG16,
    'ResNet50'    : ResNet50,
    'DenseNet121' : DenseNet121,
    'InceptionV3' : InceptionV3,
}

# Last 20% of each model's layers unfrozen for fair comparison
# VGG16       : 19  layers → 20% = 4  layers
# ResNet50    : 175 layers → 20% = 35 layers
# DenseNet121 : 427 layers → 20% = 85 layers
# InceptionV3 : 311 layers → 20% = 62 layers
FINE_TUNE_AT = {
    'VGG16'       : 4,
    'ResNet50'    : 35,
    'DenseNet121' : 85,
    'InceptionV3' : 62,
}


def build_model(model_name):
    base = BASE_MODELS[model_name](
        weights='imagenet',
        include_top=False,
        input_shape=IMG_SHAPE
    )

    # Build model first
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base.input, outputs=output)

    # Freeze layers after model is built
    num_base_layers = len(base.layers)
    fine_tune_at = FINE_TUNE_AT[model_name]
    for i, layer in enumerate(model.layers):
        if i < num_base_layers - fine_tune_at:
            layer.trainable = False
        else:
            layer.trainable = True

    frozen    = sum(1 for l in model.layers if not l.trainable)
    trainable = sum(1 for l in model.layers if l.trainable)

    # Compile after freezing
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    print(f"\n{'='*55}")
    print(f"  Model                 : {model_name}")
    print(f"  Base layers frozen    : {frozen}")
    print(f"  Base layers trainable : {trainable}")
    print(f"  Total parameters      : {model.count_params():,}")
    print(f"  Trainable parameters  : "
          f"{sum(tf.size(w).numpy() for w in model.trainable_weights):,}")
    print(f"  Non-trainable params  : "
          f"{sum(tf.size(w).numpy() for w in model.non_trainable_weights):,}")
    print(f"{'='*55}\n")

    return model


# ── Step 6: Train remaining models ─────────────────────────────────────────
import pandas as pd

all_histories = {}

# Calculate class weights to handle imbalance
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_generator.classes
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f"Class weights: {class_weight_dict}")
for model_name in ['InceptionV3']:
    print(f"\n{'#'*60}")
    print(f"  Training: {model_name}")
    print(f"{'#'*60}\n")

    model = build_model(model_name)

    callbacks = [
        ModelCheckpoint(
            filepath=f'/content/models/saved_models/{model_name}_400X_best.keras',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        EarlyStopping(
            monitor='val_loss',
            patience=7,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
        CSVLogger(
            filename=f'/content/results/metrics/{model_name}_400X_training_log.csv',
            append=False
        )
    ]

    history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

    model.save(f'/content/models/saved_models/{model_name}_400X_final.keras')
    all_histories[model_name] = history

    print(f"\n  {model_name} complete.")
    print(f"  Best val accuracy: {max(history.history['val_accuracy']):.4f}")

print("\n" + "="*60)
print("  All models trained successfully.")
print("="*60)

# ── Step 7: Plot training curves ───────────────────────────────────────────
for model_name, history in all_histories.items():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Validation')
    ax1.set_title(f'{model_name} — Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()

    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Validation')
    ax2.set_title(f'{model_name} — Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()

    plt.tight_layout()
    plt.savefig(
        f'/content/results/plots/{model_name}_400X_training_curve.png',
        dpi=150
    )
    plt.close()
    print(f"Training curve saved: {model_name}")

# ── Step 8: Copy models and results to Google Drive ────────────────────────
import shutil

drive_output = '/content/drive/MyDrive/breast_cancer/trained_models'
os.makedirs(drive_output, exist_ok=True)

for f in os.listdir('/content/models/saved_models'):
    shutil.copy(
        f'/content/models/saved_models/{f}',
        f'{drive_output}/{f}'
    )

drive_results = '/content/drive/MyDrive/breast_cancer/results'
os.makedirs(drive_results, exist_ok=True)
shutil.copytree('/content/results', drive_results, dirs_exist_ok=True)

print("\nAll models and results copied to Google Drive.")
print(f"Models  : {drive_output}")
print(f"Results : {drive_results}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Benign images    : 590
Malignant images : 1245
Total            : 1835
Found 1285 images belonging to 2 classes.
Found 550 images belonging to 2 classes.
Found 550 images belonging to 2 classes.

Train      : 1285 images
Validation : 550 images
Test       : 550 images
Classes    : {'benign': 0, 'malignant': 1}
Class weights: {0: np.float64(1.555690072639225), 1: np.float64(0.7368119266055045)}

############################################################
  Training: InceptionV3
############################################################


  Model                 : InceptionV3
  Base layers frozen    : 249
  Base layers trainable : 68
  Total parameters      : 22,360,353
  Trainable parameters  : 11,672,449
  Non-trainable params  : 10,687,904

Epoch 1/30
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5729 - loss: 0.6811
Epoch 1: val_accuracy improved fr

In [ ]:
import tensorflow as tf

MODEL_NAMES = ['VGG16', 'ResNet50', 'DenseNet121', 'InceptionV3']

for model_name in MODEL_NAMES:
    print(f"Converting {model_name}...")
    model = tf.keras.models.load_model(
        f'/content/drive/MyDrive/breast_cancer/trained_models/{model_name}_400X_best.keras'
    )
    h5_path = f'/content/drive/MyDrive/breast_cancer/trained_models/{model_name}_400X_best.h5'
    model.save(h5_path, save_format='h5')
    print(f"  Saved: {h5_path}")

print("\nAll models converted successfully.")

Converting VGG16...


ValueError: File not found: filepath=/content/drive/MyDrive/breast_cancer/trained_models/VGG16_400X_best.keras. Please ensure the file is an accessible `.keras` zip file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import tensorflow as tf

MODEL_NAMES = ['VGG16', 'ResNet50', 'DenseNet121', 'InceptionV3']

for model_name in MODEL_NAMES:
    print(f"Converting {model_name}...")
    model = tf.keras.models.load_model(
        f'/content/drive/MyDrive/breast_cancer/trained_models/{model_name}_400X_best.keras'
    )
    h5_path = f'/content/drive/MyDrive/breast_cancer/trained_models/{model_name}_400X_best.h5'
    model.save(h5_path, save_format='h5')
    print(f"  Saved: {h5_path}")

print("\nAll models converted successfully.")


Converting VGG16...


  Saved: /content/drive/MyDrive/breast_cancer/trained_models/VGG16_400X_best.h5
Converting ResNet50...


  Saved: /content/drive/MyDrive/breast_cancer/trained_models/ResNet50_400X_best.h5
Converting DenseNet121...


  Saved: /content/drive/MyDrive/breast_cancer/trained_models/DenseNet121_400X_best.h5
Converting InceptionV3...


  Saved: /content/drive/MyDrive/breast_cancer/trained_models/InceptionV3_400X_best.h5

All models converted successfully.


In [ ]:
import tensorflow as tf
import keras

# Custom Dense layer that ignores quantization_config
class CompatibleDense(keras.layers.Dense):
    def __init__(self, *args, **kwargs):
        kwargs.pop('quantization_config', None)
        super().__init__(*args, **kwargs)

MODEL_NAMES = ['VGG16', 'ResNet50', 'DenseNet121', 'InceptionV3']

for model_name in MODEL_NAMES:
    print(f"Converting {model_name}...")
    model = tf.keras.models.load_model(
        f'/content/drive/MyDrive/breast_cancer/trained_models/{model_name}_400X_best.keras',
        custom_objects={'Dense': CompatibleDense}
    )
    # Save weights only
    weights_path = f'/content/drive/MyDrive/breast_cancer/trained_models/{model_name}_400X_weights.weights.h5'
    model.save_weights(weights_path)
    print(f"  Weights saved: {weights_path}")

print("\nAll weights saved successfully.")

Converting VGG16...
  Weights saved: /content/drive/MyDrive/breast_cancer/trained_models/VGG16_400X_weights.weights.h5
Converting ResNet50...
  Weights saved: /content/drive/MyDrive/breast_cancer/trained_models/ResNet50_400X_weights.weights.h5
Converting DenseNet121...
  Weights saved: /content/drive/MyDrive/breast_cancer/trained_models/DenseNet121_400X_weights.weights.h5
Converting InceptionV3...
  Weights saved: /content/drive/MyDrive/breast_cancer/trained_models/InceptionV3_400X_weights.weights.h5

All weights saved successfully.


In [ ]:
import keras
print(keras.__version__)


3.13.2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE     = '/content/drive/MyDrive/breast_cancer'
MAGNIFICATIONS = ['40X', '100X', '200X', '400X']

for mag in MAGNIFICATIONS:
    dataset_dir = f'{DRIVE_BASE}/{mag}'
    if os.path.exists(dataset_dir):
        benign    = len(os.listdir(f'{dataset_dir}/benign'))
        malignant = len(os.listdir(f'{dataset_dir}/malignant'))
        print(f"{mag} — Benign: {benign}, Malignant: {malignant}, Total: {benign + malignant}")
    else:
        print(f"{mag} — NOT FOUND")

Mounted at /content/drive
40X — Benign: 625, Malignant: 1372, Total: 1997
100X — Benign: 644, Malignant: 1437, Total: 2081
200X — Benign: 623, Malignant: 1390, Total: 2013
400X — Benign: 590, Malignant: 1245, Total: 1835


In [2]:
# ── Step 1: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Step 2: Set up folder structure ───────────────────────────────────────
import os

os.makedirs('/content/models/saved_models', exist_ok=True)
os.makedirs('/content/results/metrics', exist_ok=True)
os.makedirs('/content/results/plots', exist_ok=True)

# ── Step 3: Imports ────────────────────────────────────────────────────────
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import (
    VGG16, ResNet50, DenseNet121, InceptionV3
)
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping,
    ReduceLROnPlateau, CSVLogger
)
from sklearn.utils.class_weight import compute_class_weight

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {tf.config.list_physical_devices('GPU')}")

# ── Step 4: Constants ──────────────────────────────────────────────────────
IMG_SIZE      = (224, 224)
IMG_SHAPE     = (224, 224, 3)
BATCH_SIZE    = 32
SEED          = 42
EPOCHS        = 30
LEARNING_RATE = 1e-4
DRIVE_BASE    = '/content/drive/MyDrive/breast_cancer'
MAGNIFICATIONS = ['40X', '100X', '200X', '400X']

BASE_MODELS = {
    'VGG16'       : VGG16,
    'ResNet50'    : ResNet50,
    'DenseNet121' : DenseNet121,
    'InceptionV3' : InceptionV3,
}

# Last 20% of layers unfrozen per model for fair comparison
FINE_TUNE_AT = {
    'VGG16'       : 4,
    'ResNet50'    : 35,
    'DenseNet121' : 85,
    'InceptionV3' : 62,
}


# ── Step 5: Data generators ────────────────────────────────────────────────
def get_data_generators(magnification: str):
    data_dir = f'{DRIVE_BASE}/{magnification}'

    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=20,
        horizontal_flip=True,
        vertical_flip=True,
        zoom_range=0.2,
        width_shift_range=0.1,
        height_shift_range=0.1,
        validation_split=0.30
    )

    val_test_datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        validation_split=0.30
    )

    train_generator = train_datagen.flow_from_directory(
        data_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='binary',
        subset='training',
        shuffle=True,
        seed=SEED
    )

    val_generator = val_test_datagen.flow_from_directory(
        data_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='binary',
        subset='validation',
        shuffle=False,
        seed=SEED
    )

    test_generator = val_test_datagen.flow_from_directory(
        data_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='binary',
        subset='validation',
        shuffle=False,
        seed=SEED + 1
    )

    return train_generator, val_generator, test_generator


# ── Step 6: Build model ────────────────────────────────────────────────────
def build_model(model_name: str):
    base = BASE_MODELS[model_name](
        weights='imagenet',
        include_top=False,
        input_shape=IMG_SHAPE
    )

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base.input, outputs=output)

    num_base_layers = len(base.layers)
    fine_tune_at    = FINE_TUNE_AT[model_name]
    for i, layer in enumerate(model.layers):
        if i < num_base_layers - fine_tune_at:
            layer.trainable = False
        else:
            layer.trainable = True

    frozen    = sum(1 for l in model.layers if not l.trainable)
    trainable = sum(1 for l in model.layers if l.trainable)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    print(f"\n{'='*55}")
    print(f"  Model                 : {model_name}")
    print(f"  Base layers frozen    : {frozen}")
    print(f"  Base layers trainable : {trainable}")
    print(f"  Total parameters      : {model.count_params():,}")
    print(f"  Trainable parameters  : "
          f"{sum(tf.size(w).numpy() for w in model.trainable_weights):,}")
    print(f"  Non-trainable params  : "
          f"{sum(tf.size(w).numpy() for w in model.non_trainable_weights):,}")
    print(f"{'='*55}\n")

    return model


# ── Step 7: Train all models across all magnifications ────────────────────
all_histories = {}

for mag in MAGNIFICATIONS:
    dataset_dir = f'{DRIVE_BASE}/{mag}'
    if not os.path.exists(dataset_dir):
        print(f"\n[SKIP] {mag} not found in Drive")
        continue

    print(f"\n{'='*60}")
    print(f"  MAGNIFICATION: {mag}")
    print(f"{'='*60}")

    train_gen, val_gen, test_gen = get_data_generators(mag)

    # Class weights for imbalance
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.array([0, 1]),
        y=train_gen.classes
    )
    class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
    print(f"  Class weights: {class_weight_dict}")

    for model_name in BASE_MODELS:
        key = f"{model_name}_{mag}"
        print(f"\n{'#'*60}")
        print(f"  Training: {model_name} | {mag}")
        print(f"{'#'*60}\n")

        model = build_model(model_name)

        callbacks = [
            ModelCheckpoint(
                filepath=f'/content/models/saved_models/{model_name}_{mag}_best.keras',
                monitor='val_accuracy',
                save_best_only=True,
                verbose=1
            ),
            EarlyStopping(
                monitor='val_loss',
                patience=7,
                restore_best_weights=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=3,
                min_lr=1e-7,
                verbose=1
            ),
            CSVLogger(
                filename=f'/content/results/metrics/{model_name}_{mag}_training_log.csv',
                append=False
            )
        ]

        history = model.fit(
            train_gen,
            epochs=EPOCHS,
            validation_data=val_gen,
            callbacks=callbacks,
            class_weight=class_weight_dict,
            verbose=1
        )

        model.save(
            f'/content/models/saved_models/{model_name}_{mag}_final.keras'
        )
        all_histories[key] = history

        best_val = max(history.history['val_accuracy'])
        print(f"\n  {key} complete.")
        print(f"  Best val accuracy: {best_val:.4f}")

        # Save to Drive immediately after each model
        import shutil
        drive_output = f'{DRIVE_BASE}/trained_models_all_magnifications'
        os.makedirs(drive_output, exist_ok=True)
        for f in os.listdir('/content/models/saved_models'):
            shutil.copy(
                f'/content/models/saved_models/{f}',
                f'{drive_output}/{f}'
            )
        print(f"  Models backed up to Drive.")

print(f"\n{'='*60}")
print("  All models trained successfully.")
print(f"{'='*60}")

# ── Step 8: Plot training curves ───────────────────────────────────────────
for key, history in all_histories.items():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Validation')
    ax1.set_title(f'{key} — Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()

    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Validation')
    ax2.set_title(f'{key} — Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()

    plt.tight_layout()
    plt.savefig(
        f'/content/results/plots/{key}_training_curve.png',
        dpi=150
    )
    plt.close()
    print(f"Training curve saved: {key}")

# ── Step 9: Save all results to Google Drive ───────────────────────────────
import shutil

drive_results = f'{DRIVE_BASE}/results_all_magnifications'
os.makedirs(drive_results, exist_ok=True)
shutil.copytree('/content/results', drive_results, dirs_exist_ok=True)

print("\nAll results copied to Google Drive.")
print(f"Models  : {DRIVE_BASE}/trained_models_all_magnifications")
print(f"Results : {drive_results}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TensorFlow version : 2.20.0
GPU available      : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

  MAGNIFICATION: 40X
Found 1399 images belonging to 2 classes.
Found 598 images belonging to 2 classes.
Found 598 images belonging to 2 classes.
  Class weights: {0: np.float64(1.5970319634703196), 1: np.float64(0.7278876170655567)}

############################################################
  Training: VGG16 | 40X
############################################################

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

  Model                 : VGG16
  Base layers frozen    : 15
  Base layers trainable : 10
  Total parameters      : 14,879,041
  Trainable parameters  : 7,243,777
  Non-trainable params  : 7,635,264

Epoch 1/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 12s/step - accuracy: 0.5539 - loss: 0.6989 
Epoch 1: val_accuracy improved from 

In [3]:
import pandas as pd
import os

MAGNIFICATIONS = ['40X', '100X', '200X', '400X']
MODEL_NAMES    = ['VGG16', 'ResNet50', 'DenseNet121', 'InceptionV3']
DRIVE_BASE     = '/content/drive/MyDrive/breast_cancer'

results = []

for mag in MAGNIFICATIONS:
    for model_name in MODEL_NAMES:
        log_path = f'/content/drive/MyDrive/breast_cancer/results_all_magnifications/metrics/{model_name}_{mag}_training_log.csv'
        if os.path.exists(log_path):
            df = pd.read_csv(log_path)
            best_val = df['val_accuracy'].max()
            best_epoch = df['val_accuracy'].idxmax() + 1
            results.append({
                'Model'       : model_name,
                'Magnification': mag,
                'Best Val Acc': round(best_val, 4),
                'Best Epoch'  : best_epoch
            })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
df_results.to_csv(f'{DRIVE_BASE}/results_all_magnifications/metrics/all_magnifications_summary.csv', index=False)
print("\nSummary saved to Drive.")

      Model Magnification  Best Val Acc  Best Epoch
      VGG16           40X        0.7826           8
   ResNet50           40X        0.6973           8
DenseNet121           40X        0.7625           9
InceptionV3           40X        0.7793           9
      VGG16          100X        0.8109           9
   ResNet50          100X        0.6891           4
DenseNet121          100X        0.8750          16
InceptionV3          100X        0.7885           6
      VGG16          200X        0.8590          13
   ResNet50          200X        0.6915           1
DenseNet121          200X        0.8856          13
InceptionV3          200X        0.8226           9
      VGG16          400X        0.8327          13
   ResNet50          400X        0.7418          17
DenseNet121          400X        0.8782          22
InceptionV3          400X        0.8273           7

Summary saved to Drive.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/eval_results/plots', exist_ok=True)
os.makedirs('/content/eval_results/metrics', exist_ok=True)
# ── Full Evaluation: All 16 Models ────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import (
    VGG16, ResNet50, DenseNet121, InceptionV3
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve
)
from sklearn.utils.class_weight import compute_class_weight
import os, shutil

DRIVE_BASE     = '/content/drive/MyDrive/breast_cancer'
MAGNIFICATIONS = ['40X', '100X', '200X', '400X']
IMG_SIZE       = (224, 224)
IMG_SHAPE      = (224, 224, 3)
BATCH_SIZE     = 32
SEED           = 42

BASE_MODELS = {
    'VGG16'       : VGG16,
    'ResNet50'    : ResNet50,
    'DenseNet121' : DenseNet121,
    'InceptionV3' : InceptionV3,
}

os.makedirs('/content/eval_results/plots', exist_ok=True)
os.makedirs('/content/eval_results/metrics', exist_ok=True)


# ── Rebuild model and load weights ────────────────────────────────────────────
def load_model(model_name, magnification):
    model_path = (
        f'{DRIVE_BASE}/trained_models_all_magnifications/'
        f'{model_name}_{magnification}_best.keras'
    )
    if not os.path.exists(model_path):
        print(f"  [SKIP] Not found: {model_path}")
        return None

    model = tf.keras.models.load_model(model_path)
    return model


# ── Get test generator ────────────────────────────────────────────────────────
def get_test_generator(magnification):
    data_dir = f'{DRIVE_BASE}/{magnification}'

    val_test_datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        validation_split=0.30
    )

    test_generator = val_test_datagen.flow_from_directory(
        data_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='binary',
        subset='validation',
        shuffle=False,
        seed=SEED + 1
    )
    return test_generator


# ── Plot confusion matrix ─────────────────────────────────────────────────────
def plot_confusion_matrix(model_name, mag, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Benign', 'Malignant'],
        yticklabels=['Benign', 'Malignant']
    )
    plt.title(f'Confusion Matrix — {model_name} | {mag}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    path = f'/content/eval_results/plots/{model_name}_{mag}_confusion_matrix.png'
    plt.savefig(path, dpi=150)
    plt.close()


# ── Plot ROC curve ────────────────────────────────────────────────────────────
def plot_roc_curve(model_name, mag, y_true, y_pred_prob):
    fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
    auc = roc_auc_score(y_true, y_pred_prob)
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, color='darkorange', label=f'AUC = {auc:.4f}')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve — {model_name} | {mag}')
    plt.legend()
    plt.tight_layout()
    path = f'/content/eval_results/plots/{model_name}_{mag}_roc_curve.png'
    plt.savefig(path, dpi=150)
    plt.close()


# ── Evaluate all models ───────────────────────────────────────────────────────
all_metrics = []

for mag in MAGNIFICATIONS:
    print(f"\n{'='*60}")
    print(f"  Evaluating: {mag}")
    print(f"{'='*60}")

    test_gen = get_test_generator(mag)

    for model_name in BASE_MODELS:
        print(f"\n  ── {model_name} | {mag} ──")

        model = load_model(model_name, mag)
        if model is None:
            continue

        test_gen.reset()
        y_pred_prob = model.predict(test_gen, verbose=1).flatten()
        y_pred      = (y_pred_prob > 0.5).astype(int)
        y_true      = test_gen.classes

        metrics = {
            'Model'         : model_name,
            'Magnification' : mag,
            'Accuracy'      : round(accuracy_score(y_true, y_pred), 4),
            'Precision'     : round(precision_score(y_true, y_pred), 4),
            'Recall'        : round(recall_score(y_true, y_pred), 4),
            'F1-Score'      : round(f1_score(y_true, y_pred), 4),
            'ROC-AUC'       : round(roc_auc_score(y_true, y_pred_prob), 4),
        }

        print(f"    Accuracy  : {metrics['Accuracy']}")
        print(f"    Precision : {metrics['Precision']}")
        print(f"    Recall    : {metrics['Recall']}")
        print(f"    F1-Score  : {metrics['F1-Score']}")
        print(f"    ROC-AUC   : {metrics['ROC-AUC']}")

        all_metrics.append(metrics)

        plot_confusion_matrix(model_name, mag, y_true, y_pred)
        plot_roc_curve(model_name, mag, y_true, y_pred_prob)

        # Free memory
        del model
        tf.keras.backend.clear_session()


# ── Save comparison table ─────────────────────────────────────────────────────
df = pd.DataFrame(all_metrics)
print("\n── Full Model Comparison ──────────────────────────────────")
print(df.to_string(index=False))

csv_path = '/content/eval_results/metrics/full_model_comparison.csv'
df.to_csv(csv_path, index=False)
print(f"\nFull comparison table saved: {csv_path}")


# ── Accuracy heatmap across models and magnifications ─────────────────────────
pivot = df.pivot(index='Magnification', columns='Model', values='Accuracy')
plt.figure(figsize=(10, 5))
sns.heatmap(
    pivot, annot=True, fmt='.4f', cmap='YlOrRd',
    linewidths=0.5
)
plt.title('Accuracy Heatmap — All Models × All Magnifications')
plt.tight_layout()
plt.savefig('/content/eval_results/plots/accuracy_heatmap.png', dpi=150)
plt.close()
print("Accuracy heatmap saved.")


# ── Bar chart comparison ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

for ax, metric in zip(axes, metrics_to_plot):
    pivot_m = df.pivot(index='Model', columns='Magnification', values=metric)
    pivot_m.plot(kind='bar', ax=ax, colormap='Set2', legend=(metric == 'Accuracy'))
    ax.set_title(metric)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=15, fontsize=8)

plt.suptitle('Model Performance Across All Magnifications', fontsize=14)
plt.tight_layout()
plt.savefig('/content/eval_results/plots/full_comparison_chart.png', dpi=150)
plt.close()
print("Full comparison chart saved.")


# ── Copy all results to Google Drive ──────────────────────────────────────────
drive_eval = f'{DRIVE_BASE}/results_all_magnifications/evaluation'
os.makedirs(drive_eval, exist_ok=True)
shutil.copytree('/content/eval_results', drive_eval, dirs_exist_ok=True)
print(f"\nAll evaluation results copied to Drive: {drive_eval}")

Mounted at /content/drive

  Evaluating: 40X
Found 598 images belonging to 2 classes.

  ── VGG16 | 40X ──
19/19 ━━━━━━━━━━━━━━━━━━━━ 249s 13s/step
    Accuracy  : 0.7826
    Precision : 0.7983
    Recall    : 0.9148
    F1-Score  : 0.8526
    ROC-AUC   : 0.8077

  ── ResNet50 | 40X ──
19/19 ━━━━━━━━━━━━━━━━━━━━ 79s 4s/step
    Accuracy  : 0.6973
    Precision : 0.6969
    Recall    : 0.9903
    F1-Score  : 0.8181
    ROC-AUC   : 0.4595

  ── DenseNet121 | 40X ──
19/19 ━━━━━━━━━━━━━━━━━━━━ 80s 4s/step
    Accuracy  : 0.7625
    Precision : 0.7918
    Recall    : 0.8881
    F1-Score  : 0.8372
    ROC-AUC   : 0.7381

  ── InceptionV3 | 40X ──
19/19 ━━━━━━━━━━━━━━━━━━━━ 61s 3s/step
    Accuracy  : 0.7793
    Precision : 0.7937
    Recall    : 0.9173
    F1-Score  : 0.851
    ROC-AUC   : 0.7498

  Evaluating: 100X
Found 624 images belonging to 2 classes.

  ── VGG16 | 100X ──
20/20 ━━━━━━━━━━━━━━━━━━━━ 249s 12s/step
    Accuracy  : 0.8109
    Precision : 0.7936
    Recall    : 0.9814
    F